# Modelling 1: Training and model selection
**Input:** the `preprocessed/` folder. Part A explains *which* algorithms were chosen and *why*. Part B trains and tunes them, then
compares them under one protocol, so every design choice is tested rather than just argued for.

A GPU isn't needed. The full run (4 classifier families × 6 setups × 5 seeds) takes about 30 minutes. Set the environment variable
`MVSA_QUICK=1` for a 1-seed test run.

# Part A: Design choices and justification

## A1. What the problem requires
Each design choice follows from a property of this specific problem:

| Property of the problem | Evidence | What the model design must do |
|---|---|---|
| **Small dataset**: ~3,600 training posts | 3,591 posts in the training split | Use few trainable parameters and strong regularisation. No end-to-end deep training |
| **High-dimensional, dense features**: 512 + 512 CLIP dimensions | 1,024 features for early fusion | Use algorithms that work well when features are many relative to samples (linear models with L2 penalty) |
| **Imbalanced classes**: neutral is only ~10% (376 / 3,591 training posts) | Class counts in the training split | Use class weighting, and **macro-F1** (not accuracy) to choose models |
| **Modalities often disagree**: 46.8% of posts | Text and image labels differ on these posts | Include a fusion method that can *see* disagreement and decide which modality to trust |
| **Fusion needs probabilities** | late and gated fusion combine class probabilities | Every classifier must output well-calibrated `predict_proba` |
| **Results must be explainable** | The fusion decision should be readable | Prefer models whose weights can be read, especially for the fusion step |
| **Results must be comparable and repeatable** | Scores need a mean and spread over seeds | Training must be cheap enough to repeat over 5 seeds and many configurations |

## A2. Frozen CLIP features + classical classifiers, not end-to-end deep learning
Many multimodal sentiment models train large networks end to end: object detectors, graph networks, cross-attention
stacks. We deliberately **don't**:
* **Overfitting.** CLIP ViT-B/32 has ~150M parameters. Fine-tuning it on 3,600 noisy posts would memorise the
  training set. Freezing it and training only a small classifier on top keeps the number of learned parameters
  in the thousands.
* **CLIP is designed for this.** The CLIP paper evaluates its features with a *linear probe*, a logistic
  regression on frozen embeddings, and shows it is competitive with fully fine-tuned models on many tasks.
  Its text and image vectors live in one shared space, so their cosine similarity directly measures agreement.
* **Cost and repeatability.** Features are extracted once. Every model below then trains in seconds, so the full
  comparison can be repeated over 5 seeds at low cost.

## A3. Classifier families
Four scikit-learn families are compared. They cover linear vs. non-linear, and margin-based vs. probabilistic vs. tree-based:

| Algorithm | Why it suits this problem | Its role in the comparison | Settings that address A1 |
|---|---|---|---|
| **Logistic regression** (multinomial, L2) | The standard linear probe for CLIP. Works well with many features and few samples. Outputs calibrated probabilities directly. Coefficients are readable | **Main model**, and also the gate (A5) | `class_weight="balanced"`, `C` tuned, inputs standardised |
| **Linear SVM** | Maximises the margin, which is robust in high dimensions. Often slightly stronger than LR on embeddings | Checks whether a margin-based linear model beats LR | `class_weight="balanced"`, `C` tuned. **Platt scaling** (`CalibratedClassifierCV`) adds the probabilities the SVM doesn't produce |
| **Random forest** | Non-linear, needs no scaling, gives feature importances. Handles mixed features well | **Non-linear reference**: tests whether non-linearity helps | `class_weight="balanced_subsample"`, `min_samples_leaf` tuned, 300 trees |
| **MLP** (one hidden layer, 256 units) | Learns interactions *between* text and image dimensions, which a linear model on concatenated features can't | Tests whether learned interactions help early fusion | L2 penalty `alpha` tuned, early stopping. MLP has no class weights, so minority classes are **oversampled** in training |

**Considered and rejected:**
* *k-nearest neighbours*: distances become less informative in 1,024 dimensions, and it gives poor probabilities.
* *Naive Bayes*: its independence assumption is badly violated by embedding dimensions. It suits word counts, but the TF-IDF baseline already covers text.
* *Gradient boosting* (XGBoost etc.): it fills the same "non-linear tree" role as the random forest, but it's slower on 1,000+ dense features and has more hyperparameters to tune on a 450-post validation set.
* *Kernel (RBF) SVM, deep fine-tuning*: more capacity than ~3,600 samples can support (A2).

## A4. Fusion setups
The question isn't just *which classifier* but *how the two modalities are combined*. Each setup tests one hypothesis:

| Setup | Input | Hypothesis it tests |
|---|---|---|
| **Baseline** | TF-IDF + colour histogram → LR | A simple reference that everything else is compared against |
| **Text only** | CLIP text embedding | How far text alone goes |
| **Image only** | CLIP image embedding | How much the image carries on its own |
| **Early fusion** | text ⊕ image (1,024-d) | One model sees both modalities at once |
| **Early + meta** | text ⊕ image ⊕ engineered features (Step 6a/6b of preprocessing) | Whether hand-crafted features add anything beyond CLIP |
| **Late fusion** | `w·p_text + (1−w)·p_image`, with `w` tuned on validation | A fixed trust level per modality, the same for every post |
| **Gated fusion** (proposed) | unimodal probabilities + agreement features → meta-classifier | Trust per post, based on whether the modalities agree |

## A5. Design of the gated fusion (the project's contribution)
Late fusion trusts text by the same amount `w` in every post. When text and image disagree, though, the right
answer depends on *which* modality is reliable for that post. The gate is a **stacking** meta-classifier that gets:

| Input | Meaning |
|---|---|
| `p_text` (3), `p_image` (3) | Each unimodal classifier's class probabilities |
| `clip_cosine` | How similar the text and image are in CLIP's shared space. Low means they're probably about different things |
| `js_divergence` | Jensen–Shannon divergence between `p_text` and `p_image`. 0 means the classifiers fully agree, 1 means they completely disagree |
| `conf_text`, `conf_image` | The highest probability of each classifier, i.e. how sure each one is |

Three implementation decisions matter:
1. **Out-of-fold training.** If the gate were trained on probabilities from unimodal models that had *already seen*
   those training posts, the probabilities would be over-confident and the gate would learn to trust them too much.
   So the gate's training inputs come from **5-fold cross-validation**: each post's probabilities come from models
   that never saw it. The gate is then applied at test time to models trained on the full training set.
2. **The gate is a logistic regression.** It has only 10 inputs, so a simple model is enough, and its coefficients
   show directly how much each agreement signal shifts each class.
3. The gate uses the **same classifier family** as its unimodal models, so each family is compared fairly.

## A6. Training protocol
* **Hyperparameters are tuned on the validation set only**, by macro-F1, over small grids (below). Small grids
  limit how much the selection can overfit a 450-post validation set. The test set is used once, at the end.
* **The final model is chosen by validation macro-F1**, never by test score.
* Each tuned setup is retrained with **5 seeds**, and results are reported as mean ± std. Seeds change
  the random forest, MLP initialisation, oversampling and the cross-validation folds (LR and SVM are deterministic,
  so their single-model rows show ± 0.000; their gated rows still vary through the folds). The split stays fixed
  so results line up with the saved preprocessing.
* **Reported:** accuracy, macro and per-class precision, recall and F1, accuracy per agreement group, and on the **hard-mismatch set**,
  how often the prediction matches the text label and how often it matches the image label.
* **Robustness:** the chosen model is re-tested with the image or the text replaced by its training-set mean
  embedding.
* **Significance:** a McNemar test checks whether gated vs. early fusion differ by more than chance.

# Part B: Implementation
## B0. Setup and loading

In [1]:
# Make the shared code in src/ importable, wherever the repository folder is
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src", "mvsa")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from mvsa.modeling import *

Loaded from /kaggle/input/datasets/<kaggle-user>/mvsa-preprocessed/preprocessed
  train  3591 posts | CLIP text 512-d, image 512-d
  val     446 posts | CLIP text 512-d, image 512-d
  test    447 posts | CLIP text 512-d, image 512-d
  hard    355 posts | CLIP text 512-d, image 512-d
Train class counts: {'negative': np.int64(1078), 'neutral': np.int64(376), 'positive': np.int64(2137)}


## B1 and B2. Classifiers, fusion setups and the agreement gate
The four classifier families (A3), their tuning grids (A6), the fusion setups (A4) and the gate (A5) are defined in
`src/mvsa/modeling.py`, so all three modelling notebooks use the same code.

## B3. Hyperparameter tuning on the validation set (A6)
For each family: choose the best grid value for each input type, then the late-fusion weight `w`,
then the gate's `C`, all by **validation macro-F1** with seed 0. The test set isn't touched here.

In [4]:
CFG = {}
for fam in FAMILIES:
    t0 = time.time()
    CFG[fam] = tune(fam)
    print(f"{fam:14s} {time.time() - t0:6.1f}s  {CFG[fam]}")
json.dump(CFG, open(os.path.join(OUT_DIR, "tuned_hyperparameters.json"), "w"), indent=1)

logreg           49.0s  {'hp': {'text': {'C': 0.01}, 'image': {'C': 0.01}, 'early': {'C': 0.01}, 'early_meta': {'C': 0.01}}, 'late_w': 0.5, 'gate_C': 0.1}
linear_svm      131.9s  {'hp': {'text': {'C': 0.001}, 'image': {'C': 0.001}, 'early': {'C': 0.001}, 'early_meta': {'C': 0.001}}, 'late_w': 0.5, 'gate_C': 0.1}
random_forest   222.0s  {'hp': {'text': {'min_samples_leaf': 5}, 'image': {'min_samples_leaf': 5}, 'early': {'min_samples_leaf': 5}, 'early_meta': {'min_samples_leaf': 5}}, 'late_w': 0.0, 'gate_C': 0.1}
mlp              93.2s  {'hp': {'text': {'alpha': 0.1}, 'image': {'alpha': 0.1}, 'early': {'alpha': 0.001}, 'early_meta': {'alpha': 0.001}}, 'late_w': 0.5, 'gate_C': 0.1}


## B4. Training and evaluation over 5 seeds
Every setup is evaluated on **validation** (used for choosing the final model), **test** (reported), and the
**hard-mismatch** set. A hard-mismatch post has no single true label, so we report how often the model sides
with the text label and how often with the image label.

In [5]:
rows, PRED = [], {}
# Baseline: TF-IDF + colour histogram, logistic regression
Xb = lambda d: sparse.hstack([d["tfidf"], sparse.csr_matrix(d["hist"])]).tocsr()
base = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000).fit(Xb(D["train"]), ytr)
for seed in SEEDS:
    r = {"family": "logreg", "setup": "baseline_tfidf_hist", "seed": seed}
    for s in ("val", "test", "hard"):
        r.update({f"{s}_{k}": v for k, v in evaluate(base.predict_proba(Xb(D[s])), D[s]).items()})
    rows.append(r)

# Every family, seed and setup is scored on validation (for choosing), test (for reporting) and hard
GATES = {}
for fam in FAMILIES:
    for seed in SEEDS:
        t0 = time.time()
        predictors, gate = build(fam, CFG[fam], seed)
        GATES[(fam, seed)] = gate
        for setup, predict in predictors.items():
            r = {"family": fam, "setup": setup, "seed": seed}
            for s in ("val", "test", "hard"):
                P = predict(D[s])
                r.update({f"{s}_{k}": v for k, v in evaluate(P, D[s]).items()})
                if s == "test":
                    PRED[(fam, setup, seed)] = P.argmax(1)
            rows.append(r)
        print(f"{fam:14s} seed {seed}: {time.time() - t0:5.1f}s")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "all_runs.csv"), index=False)

logreg         seed 0:  14.3s
logreg         seed 1:  15.2s
logreg         seed 2:  15.8s
logreg         seed 3:  14.4s
logreg         seed 4:  14.4s
linear_svm     seed 0:  25.8s
linear_svm     seed 1:  25.4s
linear_svm     seed 2:  27.4s
linear_svm     seed 3:  26.9s
linear_svm     seed 4:  28.5s
random_forest  seed 0: 137.5s
random_forest  seed 1: 136.3s
random_forest  seed 2: 138.5s
random_forest  seed 3: 135.4s
random_forest  seed 4: 135.3s
mlp            seed 0:  47.2s
mlp            seed 1:  46.2s
mlp            seed 2:  45.4s
mlp            seed 3:  48.1s
mlp            seed 4:  56.1s


## B5. Results
Test scores, mean ± std over seeds. Rows are fusion setups, columns are classifier families.
* **Accuracy** is the share of all posts predicted correctly. A majority-class guess already scores well on it here.
* **Precision** for a class: of the posts predicted as that class, the share that really are.
* **Recall** for a class: of the posts really in that class, the share the model found.
* **F1** is the harmonic mean of precision and recall, so it needs both to be high.
* **Macro** averages give each class equal weight, so the small neutral class counts as much as the others.

In [6]:
def pm(metric):
    """Format one metric as a 'mean ± std' table of setups against families."""
    g = res.groupby(["setup", "family"])[metric]
    t = (g.mean().map("{:.3f}".format) + " ± " + g.std().fillna(0).map("{:.3f}".format)).unstack()
    order = ["baseline_tfidf_hist", "text_only", "image_only", "early", "early_meta", "late", "gated"]
    return t.reindex(order)[[f for f in FAMILIES if f in t.columns]].fillna("")

print("TEST macro-F1"); print(pm("test_macro_f1").to_string())
print("\nTEST accuracy"); print(pm("test_accuracy").to_string())
print("\nTEST macro-precision"); print(pm("test_macro_precision").to_string())
print("\nTEST macro-recall"); print(pm("test_macro_recall").to_string())

TEST macro-F1
family                      logreg     linear_svm  random_forest            mlp
setup                                                                          
baseline_tfidf_hist  0.504 ± 0.000                                             
text_only            0.518 ± 0.000  0.559 ± 0.000  0.561 ± 0.008  0.571 ± 0.014
image_only           0.561 ± 0.000  0.544 ± 0.000  0.541 ± 0.009  0.595 ± 0.012
early                0.609 ± 0.000  0.616 ± 0.000  0.569 ± 0.013  0.640 ± 0.010
early_meta           0.620 ± 0.000  0.615 ± 0.000  0.631 ± 0.009  0.655 ± 0.021
late                 0.603 ± 0.000  0.574 ± 0.000  0.541 ± 0.009  0.632 ± 0.012
gated                0.596 ± 0.004  0.626 ± 0.005  0.604 ± 0.009  0.603 ± 0.014

TEST accuracy
family                      logreg     linear_svm  random_forest            mlp
setup                                                                          
baseline_tfidf_hist  0.622 ± 0.000                                             
text_only  

### Choosing the final model (by validation, not test)

In [7]:
# Model selection: the highest mean validation macro-F1 wins. Test columns are only reported, never used to choose.
summary = res.groupby(["family", "setup"]).mean(numeric_only=True).drop(columns="seed")
best_fam, best_setup = summary["val_macro_f1"].idxmax()
print(f"Chosen by validation macro-F1: {best_fam} + {best_setup}\n")
cols = ["val_macro_f1", "test_macro_f1", "test_accuracy", "test_f1_negative", "test_f1_neutral", "test_f1_positive",
        "test_acc_agreement", "test_acc_soft_mismatch", "hard_matches_text", "hard_matches_image", "hard_pred_neutral"]
top = summary.sort_values("val_macro_f1", ascending=False)[[c for c in cols if c in summary]].head(8)
print(top.round(3).to_string())
summary.round(4).to_csv(os.path.join(OUT_DIR, "summary_mean_over_seeds.csv"))

Chosen by validation macro-F1: logreg + late

                          val_macro_f1  test_macro_f1  test_accuracy  test_f1_negative  test_f1_neutral  test_f1_positive  test_acc_agreement  test_acc_soft_mismatch  hard_matches_text  hard_matches_image  hard_pred_neutral
family        setup                                                                                                                                                                                                            
logreg        late               0.619          0.603          0.696             0.676            0.352             0.780               0.759                   0.613              0.411               0.510              0.079
mlp           early_meta         0.613          0.655          0.757             0.710            0.421             0.834               0.811                   0.688              0.459               0.506              0.035
linear_svm    gated              0.613          0.626     

### Precision, recall and F1 per class for the chosen model
The first table averages over seeds. The second is scikit-learn's full report for seed 0, which adds each class's
number of test posts (`support`).

In [8]:
per_class = pd.DataFrame({m: [summary.loc[(best_fam, best_setup), f"test_{m}_{l}"] for l in LABELS]
                          for m in ("precision", "recall", "f1")}, index=LABELS)
print(f"{best_fam} + {best_setup}, mean over seeds:")
print(per_class.round(3).to_string())
per_class.round(4).to_csv(os.path.join(OUT_DIR, "per_class_metrics.csv"))
print("\nSeed 0:")
print(classification_report(D["test"]["y"], PRED[(best_fam, best_setup, 0)], labels=[0, 1, 2], target_names=LABELS, digits=3))

logreg + late, mean over seeds:
          precision  recall     f1
negative      0.653   0.701  0.676
neutral       0.275   0.487  0.352
positive      0.846   0.723  0.780

Seed 0:
              precision    recall  f1-score   support

    negative      0.653     0.701     0.676       134
     neutral      0.275     0.487     0.352        39
    positive      0.846     0.723     0.780       274

    accuracy                          0.696       447
   macro avg      0.591     0.637     0.603       447
weighted avg      0.738     0.696     0.711       447



### Per-group view for the chosen family
The key question: does the gate help on mismatched posts *without* hurting agreeing posts?

In [9]:
grp_cols = [c for c in ["test_acc_agreement", "test_acc_soft_mismatch", "hard_matches_text", "hard_matches_image"] if c in summary]
print(summary.loc[best_fam][grp_cols].reindex(["text_only", "image_only", "early", "late", "gated"]).round(3).to_string())

            test_acc_agreement  test_acc_soft_mismatch  hard_matches_text  hard_matches_image
setup                                                                                        
text_only                0.696                   0.505              0.597               0.273
image_only               0.684                   0.588              0.248               0.651
early                    0.779                   0.639              0.397               0.513
late                     0.759                   0.613              0.411               0.510
gated                    0.755                   0.569              0.376               0.494


## Save for the next notebooks

In [ ]:
save_state("run_state", {"PRED": PRED, "GATES": GATES, "best_fam": best_fam, "best_setup": best_setup, "CFG": CFG})